In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output
from beamsolve import beam as beam_module
from beamsolve import FEM


# Zone d'affichage des résultats
output = widgets.Output()


def calculate_modes(d, E, L):

    A, I = beam_module.circular_section(d=d)

    current_beam = beam_module.Beam(
        L=L,
        E=E,
        rho=7800,
        A=A,
        I=I,
        bc_left="pinned",
        bc_right="pinned"
    )

    fem_model = FEM.FEMModel(
        beam=current_beam,
        ne=20
    )

    return fem_model.solve_modes(n_modes=4)


def update(d, E, L):

    with output:
        clear_output(wait=True)
        plt.close("all")

        sol_fem = calculate_modes(d, E, L)

        print("PARAMÈTRES")
        print(f"d = {d*1000:.1f} mm")
        print(f"E = {E/1e9:.1f} GPa")
        print(f"L = {L:.2f} m")

        print("\nFRÉQUENCES PROPRES FEM")

        for k, f in enumerate(sol_fem.frequencies):
            print(f"Mode {k+1}: {f:.3f} Hz")

        FEM.visualisation.plot_mode_shapes(sol_fem)


# Curseurs
diameter = widgets.FloatSlider(
    value=0.010,
    min=0.005,
    max=0.020,
    step=0.001,
    description="d (m)",
    continuous_update=False
)

young_modulus = widgets.FloatSlider(
    value=70e9,
    min=10e9,
    max=210e9,
    step=10e9,
    description="E (Pa)",
    continuous_update=False
)

length = widgets.FloatSlider(
    value=0.50,
    min=0.20,
    max=1.00,
    step=0.05,
    description="L (m)",
    continuous_update=False
)


# Mise à jour à chaque modification
def on_change(change):
    update(
        diameter.value,
        young_modulus.value,
        length.value
    )


diameter.observe(on_change, names="value")
young_modulus.observe(on_change, names="value")
length.observe(on_change, names="value")


# Affichage initial
display(
    widgets.VBox([diameter, young_modulus, length]),
    output
)

update(
    diameter.value,
    young_modulus.value,
    length.value
)
